# N4 bias correction — tensor interface

Run `antstorch.bspline_flows.n4_bias_field_correction_tensor` on the real r16
image from the ANTsX tutorial. Inputs are PyTorch tensors plus an explicit
`ImageDomain`; the corrected image and bias field are tensors on the selected
device. ANTsPy reads the input, but the N4 call does not accept an ANTsImage.

Compare [the ANTsImage interface](n4_bias_correction_antsimage.ipynb), which uses
the same r16 preprocessing and correction settings and preserves geometry for you.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
from antstorch.bspline_flows import ImageDomain, n4_bias_field_correction_tensor

reference = ants.image_read(ants.get_ants_data("r16")).clone("float")
# Keep the real anatomy; reduce resolution only to shorten this tutorial.
reference = ants.resample_image(reference, (128, 128), use_voxels=True)
reference_mask = ants.get_mask(reference)
image = torch.from_numpy(reference.numpy().T.copy())[None, None].to(device)
mask = torch.from_numpy(reference_mask.numpy().T.copy())[None, None].to(device)
domain = ImageDomain(reference.shape, spacing=reference.spacing, origin=reference.origin,
                     direction=tuple(map(tuple, reference.direction)))
options = dict(shrink_factor=2, convergence={"iters": [10, 10], "tol": 1e-7},
               spline_param=(4, 4), number_of_histogram_bins=64)
with time_step('N4 bias correction (corrected)'):
    corrected = n4_bias_field_correction_tensor(image, domain, mask=mask, **options)
with time_step('N4 bias correction (bias)'):
    bias = n4_bias_field_correction_tensor(image, domain, mask=mask,
                                          return_bias_field=True, **options)
assert corrected.shape == image.shape and torch.isfinite(corrected).all()
assert torch.isfinite(bias).all() and (bias > 0).all()

## Inspect correction and the estimated bias field
The input has real anatomical intensity variation: a lower global coefficient
of variation is not an accuracy criterion. Inspect tissue contrast, the mask and
the estimated smooth field. No ground-truth bias field is available for r16.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, values, title in zip(axes, [image, corrected, bias],
                             ["Tutorial r16", "N4 corrected", "Estimated bias"]):
    view = ax.imshow(values[0, 0].detach().cpu(), origin="lower", cmap="gray")
    ax.set_title(title)
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

This short demonstration schedule is not a convergence prescription for every
MRI. Choose the mask, resolution, spline spacing and convergence schedule for
the intended data. The image and mask use ANTs `(x, y)` geometry, while tensors
use `(batch, channels, y, x)`.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")